In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
cars_path = os.path.join(path,"Q3_data.csv")
df= pd.read_csv(cars_path) # to read the CVS file

print(f"Dataset shape: {df.shape}") # just to get infoo about the shape

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here
null_values = df.isnull().sum()
print(null_values)

for col in df.columns:
    df[col] = df[col].fillna(df[col].mean())
print("---------------------------")
print(df.isnull().sum().sum())




In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df) # first calling to remove duplicates
check_duplicates(df) # just to check no duplicates left

In [ ]:
# Task 3: Write your code here:
# according to the info there is no catagorical columns

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

# standerzing the features
scaler = StandardScaler()
X_scaled= scaler.fit_transform(X)

print(X_scaled)

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns

def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")
#imbalanced. Use StratifiedKFold and focus on F1-score

In [ ]:
# Task 1: Write your code here:
X=X
y= y

In [ ]:
%pip install kagglehub catboost

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier
import numpy as np
lr_accuracy = []
lr_f1 = []
cat_boost = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{5}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  cat_boost.fit(X_train, y_train) # train
  y_pred = cat_boost.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  lr_accuracy.append(accuracy)
  lr_f1.append(f1)












In [ ]:
  print(f"  avg accuracy:  {np.mean(lr_accuracy):.4f}")
  print(f"  avg f1 score:  {np.mean(lr_f1):.4f}")



In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': cat_boost.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
feature_importance.iloc[0]
# the most important feature is P_2

In [ ]:
# Task Bonus: Write your code here: